# Stock Price Prediction — XGBoost + LSTM
29 tickers, 5-min OHLCV bars (2020–2025), temporal train/test split, GPU training.

In [ ]:
import os, warnings, subprocess, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

## Download Data

In [ ]:
BASE_URL = 'https://8222dad36c757d4e-184-67-221-54.serveousercontent.com'
TICKERS = [
    'AAPL','AMD','AMZN','BABA','BAC','BA','C','CSCO','CVX','DIS',
    'F','GE','GOOGL','IBM','INTC','JNJ','JPM','KO','MCD','META',
    'MSFT','NFLX','NVDA','PFE','T','TSLA','VZ','WMT','XOM'
]

os.makedirs('stock_data', exist_ok=True)
data = {}
for ticker in TICKERS:
    path = f'stock_data/{ticker}.csv'
    if not os.path.exists(path) or os.path.getsize(path) < 1024:
        subprocess.run(
            ['wget', '-q', '-c', '--tries=5', '--timeout=60',
             '--no-check-certificate', '-O', path, f'{BASE_URL}/{ticker}.csv'],
            check=True
        )
    data[ticker] = pd.read_csv(path, parse_dates=['date'])
    print(f'{ticker}: {len(data[ticker])} rows')

prices = pd.concat(
    [df.assign(ticker=ticker)[['ticker','date','open','high','low','close','volume']]
     for ticker, df in data.items()],
    ignore_index=True
)
prices = prices.sort_values(['ticker','date']).reset_index(drop=True)
print(f'\nTotal rows: {len(prices):,} | Tickers: {prices["ticker"].nunique()}')

## Feature Engineering

In [ ]:
def create_features(prices: pd.DataFrame, horizon: int = 5):
    df = prices.copy().sort_values(['ticker', 'date']).reset_index(drop=True)
    g = df.groupby('ticker', group_keys=False)

    target_col = f'target_{horizon}d'
    df[target_col] = g['close'].shift(-horizon) / df['close'] - 1.0

    df['ret_1']     = g['close'].pct_change(1)
    df['ret_5']     = g['close'].pct_change(5)
    df['ret_10']    = g['close'].pct_change(10)
    df['ret_20']    = g['close'].pct_change(20)
    df['log_ret_1'] = np.log1p(df['ret_1'])
    df['hl_spread'] = (df['high'] - df['low']) / df['close'].replace(0, np.nan)
    df['oc_spread'] = (df['close'] - df['open']) / df['open'].replace(0, np.nan)

    for w in [10, 20, 50, 100]:
        df[f'sma_{w}']      = g['close'].transform(lambda s: s.rolling(w).mean())
        df[f'ema_{w}']      = g['close'].transform(lambda s: s.ewm(span=w, adjust=False).mean())
        df[f'dist_sma_{w}'] = df['close'] / df[f'sma_{w}'] - 1.0
        df[f'dist_ema_{w}'] = df['close'] / df[f'ema_{w}'] - 1.0

    df['ema_12']      = g['close'].transform(lambda s: s.ewm(span=12, adjust=False).mean())
    df['ema_26']      = g['close'].transform(lambda s: s.ewm(span=26, adjust=False).mean())
    df['macd']        = df['ema_12'] - df['ema_26']
    df['macd_signal'] = g['macd'].transform(lambda s: s.ewm(span=9, adjust=False).mean())
    df['macd_hist']   = df['macd'] - df['macd_signal']

    for w in [10, 20, 60]:
        df[f'volatility_{w}'] = g['ret_1'].transform(lambda s: s.rolling(w).std())
        roll_std  = g['close'].transform(lambda s: s.rolling(w).std())
        roll_mean = g['close'].transform(lambda s: s.rolling(w).mean())
        df[f'bb_{w}_width'] = (4.0 * roll_std) / roll_mean

    delta    = g['close'].diff()
    avg_gain = delta.clip(lower=0).groupby(df['ticker']).transform(lambda s: s.rolling(14).mean())
    avg_loss = (-delta.clip(upper=0)).groupby(df['ticker']).transform(lambda s: s.rolling(14).mean())
    df['rsi_14'] = 100 - (100 / (1 + avg_gain / avg_loss.replace(0, np.nan)))

    df['vol_ma_20']    = g['volume'].transform(lambda s: s.rolling(20).mean())
    df['vol_ratio_20'] = df['volume'] / df['vol_ma_20']
    for w in [5, 20, 60]:
        df[f'vol_change_{w}'] = g['volume'].pct_change(w)

    df['rolling_max_10']   = g['high'].transform(lambda s: s.rolling(10).max())
    df['rolling_min_10']   = g['low'].transform(lambda s: s.rolling(10).min())
    df['dist_roll_max_10'] = df['close'] / df['rolling_max_10'] - 1.0
    df['dist_roll_min_10'] = df['close'] / df['rolling_min_10'] - 1.0

    enc = LabelEncoder()
    df['ticker_id'] = enc.fit_transform(df['ticker'].astype(str))

    feature_cols = [c for c in df.columns if c not in {'ticker', 'date', target_col}]
    dataset = df.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)

    float_cols = dataset.select_dtypes(include=['float64']).columns
    dataset[float_cols] = dataset[float_cols].astype('float32')
    dataset['ticker_id'] = dataset['ticker_id'].astype('int32')

    return dataset, feature_cols, target_col, enc

## Build Dataset & Split

In [ ]:
TARGET_HORIZON = 78  # 78 x 5-min bars = 1 full trading day ahead

dataset, feature_cols, target_col, encoder = create_features(prices, TARGET_HORIZON)

# Temporal split — 80% train / 20% test, no leakage
split_date = dataset['date'].quantile(0.80)
train_df = dataset[dataset['date'] <= split_date]
test_df  = dataset[dataset['date'] >  split_date]

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test,  y_test  = test_df[feature_cols],  test_df[target_col]

print(f'Target     : {target_col} (raw 1-day forward return)')
print(f'Split date : {pd.Timestamp(split_date).date()}')
print(f'Train rows : {len(X_train):,}')
print(f'Test rows  : {len(X_test):,}')
print(f'Features   : {len(feature_cols)}')

## XGBoost Training

In [ ]:
def clean(X, y):
    X_c = X.replace([np.inf, -np.inf], np.nan)
    y_c = y.replace([np.inf, -np.inf], np.nan)
    mask = X_c.notna().all(axis=1) & y_c.notna()
    return X_c[mask], y_c[mask]

X_train_c, y_train_c = clean(X_train, y_train)
X_test_c,  y_test_c  = clean(X_test,  y_test)

print(f'Train: {len(X_train_c):,} rows | Test: {len(X_test_c):,} rows')

xgb = XGBRegressor(
    n_estimators=5000,
    learning_rate=0.005,
    max_depth=6,
    min_child_weight=20,
    subsample=0.7,
    colsample_bytree=0.6,
    colsample_bylevel=0.6,
    reg_alpha=1.0,
    reg_lambda=5.0,
    objective='reg:squarederror',
    eval_metric='rmse',
    tree_method='hist',
    device='cuda',
    random_state=42,
    early_stopping_rounds=100,
)

xgb.fit(X_train_c, y_train_c, eval_set=[(X_test_c, y_test_c)], verbose=100)

pred_xgb = xgb.predict(X_test_c)
print(f'\nBest round   : {xgb.best_iteration}')
print(f'XGBoost RMSE : {np.sqrt(mean_squared_error(y_test_c, pred_xgb)):.6f}')
print(f'XGBoost MAE  : {mean_absolute_error(y_test_c, pred_xgb):.6f}')
print(f'Dir Accuracy : {np.mean(np.sign(pred_xgb) == np.sign(y_test_c)):.4f}')

## Analysis & Charts

In [ ]:
from scipy.stats import spearmanr

results = test_df.loc[X_test_c.index].copy()
results['pred']   = pred_xgb
results['actual'] = y_test_c.values
results['date']   = pd.to_datetime(results['date'])

fig, axes = plt.subplots(3, 2, figsize=(16, 16))

# 1. Feature Importance
importance = pd.Series(xgb.feature_importances_, index=feature_cols)
importance.nlargest(20).sort_values().plot(kind='barh', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Top 20 Feature Importances')

# 2. Per-ticker Directional Accuracy
ticker_acc = (
    results.groupby('ticker')
    .apply(lambda df: np.mean(np.sign(df['pred']) == np.sign(df['actual'])))
    .sort_values().rename('dir_accuracy')
)
ticker_acc.plot(kind='bar', ax=axes[0,1], color='steelblue')
axes[0,1].axhline(0.5, color='red', linestyle='--', label='Random (0.50)')
axes[0,1].set_title('Directional Accuracy per Ticker')
axes[0,1].legend()
axes[0,1].tick_params(axis='x', rotation=45)

# 3. Predicted vs Actual Scatter
sample = results.sample(min(20000, len(results)), random_state=42)
axes[1,0].scatter(sample['actual'], sample['pred'], alpha=0.05, s=1, color='steelblue')
axes[1,0].axhline(0, color='gray', linewidth=0.5)
axes[1,0].axvline(0, color='gray', linewidth=0.5)
lim = max(abs(sample['actual'].quantile(0.01)), abs(sample['actual'].quantile(0.99)))
axes[1,0].set_xlim(-lim, lim); axes[1,0].set_ylim(-lim, lim)
axes[1,0].set_xlabel('Actual'); axes[1,0].set_ylabel('Predicted')
axes[1,0].set_title('Predicted vs Actual')

# 4. Rolling 30-Day Directional Accuracy
res_s = results.sort_values('date').copy()
res_s['correct'] = (np.sign(res_s['pred']) == np.sign(res_s['actual'])).astype(float)
res_s.set_index('date')['correct'].rolling('30D').mean().plot(ax=axes[1,1], color='steelblue', linewidth=0.8)
axes[1,1].axhline(0.5, color='red', linestyle='--', label='Random')
axes[1,1].set_title('Rolling 30-Day Directional Accuracy')
axes[1,1].legend()

# 5. Daily IC over Time
ic_by_date = []
for date, grp in results.groupby(results['date'].dt.date):
    if len(grp) < 5: continue
    ic, _ = spearmanr(grp['pred'], grp['actual'])
    ic_by_date.append({'date': pd.Timestamp(date), 'ic': ic})
ic_df = pd.DataFrame(ic_by_date).set_index('date').sort_index()
ic_df['ic_30d'] = ic_df['ic'].rolling(30).mean()
ic_df['ic'].plot(ax=axes[2,0], alpha=0.3, color='steelblue', label='Daily IC')
ic_df['ic_30d'].plot(ax=axes[2,0], color='navy', linewidth=1.5, label='30d MA')
axes[2,0].axhline(0, color='red', linestyle='--')
axes[2,0].set_title(f'IC  Mean={ic_df["ic"].mean():.4f}  ICIR={ic_df["ic"].mean()/ic_df["ic"].std():.4f}')
axes[2,0].legend()

# 6. Long Top-5 / Short Bottom-5 Cumulative Return
ls_returns = []
for date, grp in results.groupby(results['date'].dt.date):
    if len(grp) < 10: continue
    grp_s = grp.sort_values('pred')
    ls_returns.append({'date': pd.Timestamp(date),
                       'ls_return': grp_s.tail(5)['actual'].mean() - grp_s.head(5)['actual'].mean()})
ls_df = pd.DataFrame(ls_returns).set_index('date').sort_index()
ls_df['cumulative'] = (1 + ls_df['ls_return']).cumprod()
ls_df['cumulative'].plot(ax=axes[2,1], color='steelblue', linewidth=1.2)
axes[2,1].axhline(1.0, color='red', linestyle='--', label='Baseline')
axes[2,1].set_title('Long Top-5 / Short Bottom-5 Cumulative Return')
axes[2,1].legend()

plt.suptitle('Global XGBoost Model — Full Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n{'='*45}")
print(f'  Best round   : {xgb.best_iteration}')
print(f'  RMSE         : {np.sqrt(mean_squared_error(y_test_c, pred_xgb)):.6f}')
print(f'  MAE          : {mean_absolute_error(y_test_c, pred_xgb):.6f}')
print(f'  Dir Accuracy : {np.mean(np.sign(pred_xgb) == np.sign(y_test_c)):.4f}')
print(f'  Mean IC      : {ic_df["ic"].mean():.4f}  (>0.05 good)')
print(f'  ICIR         : {ic_df["ic"].mean()/ic_df["ic"].std():.4f}  (>0.5 good)')
print(f'  L/S Total    : {ls_df["cumulative"].iloc[-1]:.4f}x')
print(f"{'='*45}")
print('\nPer-ticker Directional Accuracy:')
print(ticker_acc.sort_values(ascending=False).to_string())

## Save Model

In [ ]:
os.makedirs('model', exist_ok=True)

# Save via joblib (avoids XGBoost sklearn wrapper bug)
joblib.dump(xgb, 'model/xgb_global.pkl')

meta = {
    'feature_cols': feature_cols,
    'target_col':   target_col,
    'horizon':      TARGET_HORIZON,
    'split_date':   str(pd.Timestamp(split_date).date()),
    'tickers':      TICKERS,
    'best_round':   xgb.best_iteration,
}
with open('model/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

joblib.dump(encoder, 'model/encoder.pkl')

print('Saved:')
print('  model/xgb_global.pkl  — XGBoost model')
print('  model/meta.json       — feature list + metadata')
print('  model/encoder.pkl     — ticker label encoder')

## Inference

In [ ]:
# ── Load ──────────────────────────────────────────────────────────
model_infer   = joblib.load('model/xgb_global.pkl')
encoder_infer = joblib.load('model/encoder.pkl')

with open('model/meta.json') as f:
    meta = json.load(f)

feature_cols = meta['feature_cols']
print(f"Loaded | horizon={meta['horizon']} bars | features={len(feature_cols)}")

# ── Inference function ────────────────────────────────────────────
def predict(prices_df: pd.DataFrame) -> pd.DataFrame:
    """
    Input : DataFrame [ticker, date, open, high, low, close, volume]
            Needs >= 100 bars of history per ticker.
    Output: DataFrame [ticker, date, predicted_return, rank]
            rank=1 is the top pick for next day.
    """
    df = prices_df.copy().sort_values(['ticker','date']).reset_index(drop=True)
    g  = df.groupby('ticker', group_keys=False)

    df['ret_1']     = g['close'].pct_change(1)
    df['ret_5']     = g['close'].pct_change(5)
    df['ret_10']    = g['close'].pct_change(10)
    df['ret_20']    = g['close'].pct_change(20)
    df['log_ret_1'] = np.log1p(df['ret_1'])
    df['hl_spread'] = (df['high'] - df['low']) / df['close'].replace(0, np.nan)
    df['oc_spread'] = (df['close'] - df['open']) / df['open'].replace(0, np.nan)
    for w in [10, 20, 50, 100]:
        df[f'sma_{w}']      = g['close'].transform(lambda s: s.rolling(w).mean())
        df[f'ema_{w}']      = g['close'].transform(lambda s: s.ewm(span=w, adjust=False).mean())
        df[f'dist_sma_{w}'] = df['close'] / df[f'sma_{w}'] - 1.0
        df[f'dist_ema_{w}'] = df['close'] / df[f'ema_{w}'] - 1.0
    df['ema_12']      = g['close'].transform(lambda s: s.ewm(span=12, adjust=False).mean())
    df['ema_26']      = g['close'].transform(lambda s: s.ewm(span=26, adjust=False).mean())
    df['macd']        = df['ema_12'] - df['ema_26']
    df['macd_signal'] = g['macd'].transform(lambda s: s.ewm(span=9, adjust=False).mean())
    df['macd_hist']   = df['macd'] - df['macd_signal']
    for w in [10, 20, 60]:
        df[f'volatility_{w}'] = g['ret_1'].transform(lambda s: s.rolling(w).std())
        df[f'bb_{w}_width']   = (4 * g['close'].transform(lambda s: s.rolling(w).std())) / g['close'].transform(lambda s: s.rolling(w).mean())
    delta    = g['close'].diff()
    avg_gain = delta.clip(lower=0).groupby(df['ticker']).transform(lambda s: s.rolling(14).mean())
    avg_loss = (-delta.clip(upper=0)).groupby(df['ticker']).transform(lambda s: s.rolling(14).mean())
    df['rsi_14']       = 100 - (100 / (1 + avg_gain / avg_loss.replace(0, np.nan)))
    df['vol_ma_20']    = g['volume'].transform(lambda s: s.rolling(20).mean())
    df['vol_ratio_20'] = df['volume'] / df['vol_ma_20']
    for w in [5, 20, 60]:
        df[f'vol_change_{w}'] = g['volume'].pct_change(w)
    df['rolling_max_10']   = g['high'].transform(lambda s: s.rolling(10).max())
    df['rolling_min_10']   = g['low'].transform(lambda s: s.rolling(10).min())
    df['dist_roll_max_10'] = df['close'] / df['rolling_max_10'] - 1.0
    df['dist_roll_min_10'] = df['close'] / df['rolling_min_10'] - 1.0

    known = set(encoder_infer.classes_)
    df['ticker_id'] = df['ticker'].apply(
        lambda t: encoder_infer.transform([t])[0] if t in known else -1
    ).astype('int32')

    latest = df.sort_values('date').groupby('ticker').tail(1).copy()
    latest = latest.replace([np.inf, -np.inf], np.nan).dropna(subset=feature_cols)

    latest['predicted_return'] = model_infer.predict(latest[feature_cols].astype('float32'))
    latest['rank'] = latest['predicted_return'].rank(ascending=False).astype(int)

    return latest[['ticker','date','predicted_return','rank']].sort_values('rank')


# ── Run on latest data ────────────────────────────────────────────
preds = predict(prices)
print('\nPredictions — rank 1 = top pick for next day:')
print(preds.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# ── 1. Predicted Return Bar Chart (all stocks ranked) ─────────────
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in preds['predicted_return']]
axes[0].barh(preds['ticker'][::-1], preds['predicted_return'][::-1], color=colors[::-1])
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Predicted Next-Day Return\n(rank 1 = top pick)', fontweight='bold')
axes[0].set_xlabel('Predicted Return')

# ── 2. Rank Badge — traffic light style ───────────────────────────
n = len(preds)
rank_colors = []
for r in preds['rank']:
    if r <= n // 3:
        rank_colors.append('#2ecc71')       # top third — green
    elif r <= 2 * n // 3:
        rank_colors.append('#f39c12')       # middle — orange
    else:
        rank_colors.append('#e74c3c')       # bottom third — red

axes[1].barh(preds['ticker'][::-1], preds['rank'][::-1], color=rank_colors[::-1])
axes[1].invert_xaxis()
axes[1].set_title('Stock Rank\n(1 = best, 29 = worst)', fontweight='bold')
axes[1].set_xlabel('Rank')
axes[1].axvline(n // 3,     color='green',  linestyle='--', alpha=0.5)
axes[1].axvline(2 * n // 3, color='orange', linestyle='--', alpha=0.5)

# ── 3. Recent close price + prediction arrow per stock ────────────
# Show last 10 days of close for top-5 and bottom-5 stocks
top5    = preds.head(5)['ticker'].tolist()
bottom5 = preds.tail(5)['ticker'].tolist()
watch   = top5 + bottom5

ax3 = axes[2]
for ticker in watch:
    df_t = prices[prices['ticker'] == ticker].sort_values('date').tail(78 * 10)  # last 10 days
    # Resample to daily close
    daily = df_t.set_index('date')['close'].resample('D').last().dropna()
    pred_val = preds[preds['ticker'] == ticker]['predicted_return'].values[0]
    color = '#2ecc71' if ticker in top5 else '#e74c3c'
    label = f"{ticker} ({'▲' if pred_val > 0 else '▼'}{abs(pred_val)*100:.2f}%)"
    ax3.plot(daily.index, daily.values / daily.values[0], color=color,
             alpha=0.8, linewidth=1.5, label=label)

ax3.axhline(1.0, color='gray', linestyle='--', linewidth=0.5)
ax3.set_title('Top-5 (green) vs Bottom-5 (red)\nNormalized Price — Last 10 Days', fontweight='bold')
ax3.set_ylabel('Normalized Price')
ax3.legend(fontsize=7, loc='upper left')
ax3.tick_params(axis='x', rotation=30)

plt.suptitle(f'Stock Predictions — {preds["date"].max().date()}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# All 29 stocks — individual price chart + 1-day prediction
import math

n_stocks = len(preds)
cols = 5
rows = math.ceil(n_stocks / cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 3.5))
axes = axes.flatten()

for i, (_, row) in enumerate(preds.iterrows()):
    ticker   = row['ticker']
    pred_ret = row['predicted_return']
    rank     = int(row['rank'])
    ax       = axes[i]

    # Last 30 trading days of 5-min bars resampled to daily
    df_t  = prices[prices['ticker'] == ticker].sort_values('date').tail(78 * 30)
    daily = df_t.set_index('date')['close'].resample('D').last().dropna()

    if len(daily) < 2:
        ax.set_visible(False)
        continue

    # Colour by prediction direction
    color = '#2ecc71' if pred_ret > 0 else '#e74c3c'

    ax.plot(daily.index, daily.values, color=color, linewidth=1.5)
    ax.fill_between(daily.index, daily.values, daily.values.min(), alpha=0.15, color=color)

    # Annotate last close + predicted next close
    last_close = daily.values[-1]
    pred_close = last_close * (1 + pred_ret)
    ax.plot([daily.index[-1], daily.index[-1] + pd.Timedelta(days=1)],
            [last_close, pred_close],
            color=color, linewidth=2, linestyle='--', marker='o', markersize=4)

    # Title: ticker + rank + predicted %
    arrow = '▲' if pred_ret > 0 else '▼'
    ax.set_title(
        f'{ticker}  #{rank}\n{arrow} {pred_ret*100:+.2f}%',
        fontsize=9, fontweight='bold', color=color
    )
    ax.tick_params(axis='x', labelsize=6, rotation=30)
    ax.tick_params(axis='y', labelsize=6)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:.0f}'))
    ax.set_xlim(daily.index[0], daily.index[-1] + pd.Timedelta(days=3))

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('All Stocks — Last 30 Days + 1-Day Prediction (dashed)\nGreen = bullish  |  Red = bearish',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
